# Phase 1: Exploratory Data Analysis
**Colombo Stock Exchange (CSE) Market Analysis**
- **Date**: 2026-06-30
- **Goal**: Analyze historical stock market data (1991-2025) to identify market trends, data quality issues, and statistical properties of returns.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import scipy.stats as stats
import statsmodels.api as sm

# Import custom utilities
from utils.data_loader import (
    load_all_daily_prices,
    load_market_indices,
    load_market_stats,
    load_securities_list,
    load_splits,
    load_listings,
    load_gics,
    load_sector_market_cap,
    load_dividends,
    load_sector_ratios,
    BASE_PATH
)
from utils.plot_helpers import (
    setup_plotting_style,
    CSE_COLORS,
    SECTOR_PALETTE,
    format_large_number,
    add_crash_bands,
    CRASH_PERIODS
)

# Apply styling
setup_plotting_style()
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

## Section 1: Data Loading & Consolidation
Loading the daily prices across two eras: 1991-2000 (Close price only) and 2001-2025 (OHLCV). Supplementary datasets are also loaded.

In [ ]:
# Load consolidated daily prices (uses Parquet cache if available)
print("Loading daily prices...")
df = load_all_daily_prices()

# Load supplementary data
print("Loading supplementary data...")
df_indices = load_market_indices(BASE_PATH)
df_market_stats = load_market_stats(BASE_PATH)
try:
    df_securities = load_securities_list(BASE_PATH)
except Exception as e:
    print(f"Failed to load securities: {e}")
    df_securities = pd.DataFrame()

df_splits = load_splits(BASE_PATH)
df_listings = load_listings(BASE_PATH)
try:
    df_gics = load_gics(BASE_PATH)
except Exception as e:
    print(f"Failed to load GICS: {e}")
    df_gics = pd.DataFrame()

print("\nData Loading Complete!")
print(f"Daily Prices Shape: {df.shape}")
print(f"Date Range: {df['Date'].min().date()} to {df['Date'].max().date()}")

> **Insight - Data Structure**: The dataset spans ~34 years. The early era lacks volume and intraday price data, necessitating care when combining it with modern data.

## Section 2: Dataset Inspection
Analyzing shape, structure, and missing values across time.

In [ ]:
print("Dataset Info:")
df.info()

# Memory usage
mem_mb = df.memory_usage(deep=True).sum() / 1e6
print(f"\nMemory Usage: {mem_mb:.2f} MB")

# Number of unique companies
num_companies = df['CompanyCode'].nunique()
print(f"\nUnique Companies: {num_companies}")

# Date range per stock
stock_dates = df.groupby('CompanyCode')['Date'].agg(['min', 'max', 'count']).reset_index()
stock_dates['years_active'] = (stock_dates['max'] - stock_dates['min']).dt.days / 365.25
stock_dates = stock_dates.sort_values('years_active', ascending=False)
display(stock_dates.head(10))

plt.figure(figsize=(10, 5))
sns.histplot(stock_dates['years_active'], bins=20, color=CSE_COLORS['primary'])
plt.title('Distribution of Stock History Lengths')
plt.xlabel('Years Active')
plt.ylabel('Number of Companies')
plt.tight_layout()
plt.show()

> **Insight - Inspection**: Many companies have very short histories (recently listed or delisted), while a core set has traded for over 20 years.

## Section 3: Data Quality Assessment
Identifying missing data, zeros, and potential anomalies.

In [ ]:
# Missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Missing': missing, 'Percent': missing_pct}).sort_values('Percent', ascending=False)
display(missing_df[missing_df['Percent'] > 0])

# Zero prices
zeros = df[df['Close'] <= 0]
print(f"\nRecords with Close <= 0: {len(zeros)}")

# Duplicate dates per company
dups = df[df.duplicated(subset=['CompanyCode', 'Date'], keep=False)]
print(f"Duplicate records: {len(dups)}")

# Calculate daily return to flag anomalies
df.sort_values(['CompanyCode', 'Date'], inplace=True)
df['DailyReturn'] = df.groupby('CompanyCode')['Close'].pct_change()

# Anomaly flags
anomalies = df[abs(df['DailyReturn']) > 0.5]
print(f"\nAnomalies (|Daily Return| > 50%): {len(anomalies)} records")
display(anomalies[['CompanyCode', 'Date', 'Close', 'DailyReturn']].head(10))

> **Insight - Data Quality**: The early data (1991-2000) shows missing values for volume and OHLC. Extreme price jumps (>50%) might represent stock splits or data entry errors, which need adjustment before modeling.

## Section 4: Descriptive Statistics
Understanding the distribution of prices and liquidity.

In [ ]:
# Numerical summary
display(df.describe())

# Per stock summary
stock_summary = df.groupby('CompanyCode').agg({
    'Close': ['mean', 'std', 'min', 'max'],
    'ShareVolume': 'sum',
    'Turnover': 'sum'
})
stock_summary.columns = ['_'.join(col) for col in stock_summary.columns]

# Top 20 most actively traded
top_20_vol = stock_summary.sort_values('ShareVolume_sum', ascending=False).head(20)

plt.figure(figsize=(12, 6))
sns.barplot(x=top_20_vol['ShareVolume_sum'], y=top_20_vol.index, palette=SECTOR_PALETTE[:20])
plt.title('Top 20 Most Actively Traded Stocks (by Volume)')
plt.xlabel('Total Share Volume')
plt.ylabel('Company Code')
plt.tight_layout()
plt.show()

# Price level distribution (latest price)
latest_prices = df.groupby('CompanyCode')['Close'].last()
bins = [0, 10, 50, 100, 500, np.inf]
labels = ['< 10', '10-50', '50-100', '100-500', '500+']
price_cats = pd.cut(latest_prices, bins=bins, labels=labels).value_counts().sort_index()

plt.figure(figsize=(8, 4))
price_cats.plot(kind='bar', color=CSE_COLORS['accent1'])
plt.title('Distribution of Latest Price Levels')
plt.xlabel('Price Range (Rs.)')
plt.ylabel('Number of Companies')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

> **Insight - Descriptive Stats**: The market is highly skewed, with trading activity concentrated in a few liquid stocks. The majority of companies trade under Rs. 100.

## Section 5: Time Series Analysis: Market Indices
Analyzing ASPI and S&P SL20 performance.

In [ ]:
if not df_indices.empty and 'ASPI' in df_indices.columns:
    df_indices['Date'] = pd.to_datetime(df_indices['Date'], errors='coerce')
    df_indices.dropna(subset=['Date', 'ASPI'], inplace=True)
    df_indices.sort_values('Date', inplace=True)
    df_indices.set_index('Date', inplace=True)

    fig, ax = plt.subplots(figsize=(14, 6))
    ax.plot(df_indices.index, df_indices['ASPI'], color=CSE_COLORS['primary'], label='ASPI')
    
    df_indices['ASPI_MA90'] = df_indices['ASPI'].rolling(90).mean()
    ax.plot(df_indices.index, df_indices['ASPI_MA90'], color=CSE_COLORS['orange'], label='90-day MA', alpha=0.8)
    
    add_crash_bands(ax)
    
    ax.set_title('ASPI Performance (1985-2025) with Major Market Events')
    ax.set_ylabel('Index Value')
    ax.legend()
    plt.tight_layout()
    plt.show()
    
    # Returns
    df_indices['Daily_Return'] = df_indices['ASPI'].pct_change()
    
    monthly_aspi = df_indices['ASPI'].resample('M').last()
    monthly_returns = monthly_aspi.pct_change()
    
    annual_aspi = df_indices['ASPI'].resample('Y').last()
    annual_returns = annual_aspi.pct_change()
    
    fig, axes = plt.subplots(2, 1, figsize=(14, 10))
    axes[0].plot(df_indices.index, df_indices['Daily_Return'], color=CSE_COLORS['accent1'], linewidth=0.5)
    axes[0].set_title('ASPI Daily Returns')
    axes[0].axhline(0, color='black', linewidth=1)
    
    axes[1].bar(annual_returns.index.year, annual_returns, color=[CSE_COLORS['positive'] if x > 0 else CSE_COLORS['negative'] for x in annual_returns])
    axes[1].set_title('ASPI Annual Returns')
    axes[1].axhline(0, color='black', linewidth=1)
    
    plt.tight_layout()
    plt.show()
else:
    print("Market indices data not available.")

> **Insight - Indices**: The ASPI shows significant long-term growth punctuated by severe drawdowns during economic and political crises. Volatility is elevated during these events.

## Section 6: Time Series Analysis: Trading Volume
Assessing market liquidity trends.

In [ ]:
if not df_market_stats.empty and 'TURNOVER EQUITY-Mn' in df_market_stats.columns:
    df_market_stats['Date'] = pd.to_datetime(df_market_stats['Date'], errors='coerce')
    df_market_stats.dropna(subset=['Date'], inplace=True)
    df_market_stats.set_index('Date', inplace=True)
    
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.plot(df_market_stats.index, df_market_stats['TURNOVER EQUITY-Mn'], color=CSE_COLORS['teal'], alpha=0.5, label='Daily Turnover')
    
    turnover_ma90 = df_market_stats['TURNOVER EQUITY-Mn'].rolling(90).mean()
    ax.plot(df_market_stats.index, turnover_ma90, color=CSE_COLORS['primary'], label='90-day MA', linewidth=2)
    
    ax.set_title('Market Daily Turnover (Equity) in LKR Millions')
    ax.set_ylabel('Turnover (Mn)')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Market stats data not available.")

> **Insight - Volume**: Trading volume shows clear regimes, with massive spikes during bull markets (e.g., post-war 2010, pandemic recovery 2021) and depressed liquidity during bear phases.

## Section 7: Time Series Analysis: Individual Stocks
Analyzing top stocks over time.

In [ ]:
top_10 = top_20_vol.head(10).index.tolist()
df_modern = df[df['Era'] == '2001-2025'].copy()

fig, axes = plt.subplots(5, 2, figsize=(15, 20))
axes = axes.flatten()

for i, stock in enumerate(top_10):
    stock_data = df_modern[df_modern['CompanyCode'] == stock].set_index('Date')
    if len(stock_data) > 0:
        axes[i].plot(stock_data.index, stock_data['Close'], color=CSE_COLORS['primary'], label='Close')
        axes[i].plot(stock_data.index, stock_data['Close'].rolling(90).mean(), color=CSE_COLORS['orange'], label='90d MA')
        axes[i].set_title(f'{stock} Price History')
        axes[i].legend()

plt.tight_layout()
plt.show()

> **Insight - Stocks**: Top tier stocks show robust trend-following characteristics, but they also experience sharp pullbacks. Moving averages (90-day) effectively smooth the noise.

## Section 8: Distribution Analysis
Understanding the statistical properties of stock returns.

In [ ]:
# Market-wide daily return distribution
df_returns = df.dropna(subset=['DailyReturn'])
returns = df_returns['DailyReturn'].values

plt.figure(figsize=(10, 6))
sns.histplot(returns, bins=100, kde=True, stat="density", color=CSE_COLORS['accent1'], 
             range=(-0.15, 0.15))

# Fit normal distribution
mu, std = stats.norm.fit(returns[~np.isnan(returns)])
xmin, xmax = plt.xlim()
x = np.linspace(xmin, xmax, 100)
p = stats.norm.pdf(x, mu, std)
plt.plot(x, p, 'k', linewidth=2, label=f'Normal Fit ($\mu={mu:.4f}, \sigma={std:.4f}$)')

plt.title('Distribution of Daily Returns vs Normal Distribution')
plt.xlabel('Daily Return')
plt.ylabel('Density')
plt.legend()
plt.tight_layout()
plt.show()

# Skewness and Kurtosis
print(f"Skewness: {stats.skew(returns[~np.isnan(returns)]):.4f}")
print(f"Kurtosis: {stats.kurtosis(returns[~np.isnan(returns)]):.4f}")

> **Insight - Distributions**: The returns distribution exhibits significant excess kurtosis (fat tails) compared to a normal distribution. This implies extreme market moves happen more frequently than a normal model would predict.

## Section 9: Correlation Analysis
Analyzing relationships between price, volume, and cross-stock correlations.

In [ ]:
df_modern = df[df['Era'] == '2001-2025'].copy()
numeric_cols = ['Open', 'High', 'Low', 'Close', 'TradeVolume', 'ShareVolume', 'Turnover']

corr_matrix = df_modern[numeric_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1, center=0)
plt.title('Correlation Matrix of Price & Volume Features')
plt.tight_layout()
plt.show()

# Cross-stock correlation for top 10 stocks
top_10 = df_modern.groupby('CompanyCode')['ShareVolume'].sum().nlargest(10).index
pivot_returns = df_modern[df_modern['CompanyCode'].isin(top_10)].pivot(index='Date', columns='CompanyCode', values='DailyReturn')

plt.figure(figsize=(10, 8))
sns.heatmap(pivot_returns.corr(), annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Cross-Stock Return Correlation (Top 10)')
plt.tight_layout()
plt.show()

> **Insight - Correlations**: While Open/High/Low/Close are perfectly correlated, volume metrics have weak linear correlations with price. Cross-stock correlations are generally positive but moderate, indicating sector-specific movements and diversification potential.

## Section 10: Outlier Analysis
Analyzing extreme market events. 

*Note: In financial time series, outliers often represent actual market crashes, rallies, or news events rather than "bad data", and should generally be retained for tail-risk modeling.*

In [ ]:
Q1 = df_returns['DailyReturn'].quantile(0.25)
Q3 = df_returns['DailyReturn'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 3 * IQR
upper_bound = Q3 + 3 * IQR

outliers = df_returns[(df_returns['DailyReturn'] < lower_bound) | (df_returns['DailyReturn'] > upper_bound)]
print(f"Total Outliers detected (3x IQR): {len(outliers)} ({len(outliers)/len(df_returns)*100:.2f}%)")

plt.figure(figsize=(12, 5))
sns.boxplot(x=df_returns['DailyReturn'], color=CSE_COLORS['accent2'])
plt.axvline(lower_bound, color='red', linestyle='--')
plt.axvline(upper_bound, color='red', linestyle='--')
plt.title('Daily Returns Boxplot with Outlier Bounds')
plt.xlim(-0.2, 0.2)
plt.tight_layout()
plt.show()

# Outliers over time
outliers['Year'] = outliers['Date'].dt.year
outlier_counts = outliers.groupby('Year').size()

plt.figure(figsize=(12, 5))
outlier_counts.plot(kind='bar', color=CSE_COLORS['negative'])
plt.title('Number of Extreme Return Events per Year')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

> **Insight - Outliers**: Outlier frequency spikes during specific years (e.g., 2011, 2021) matching volatile market regimes.

## Section 11: Feature Relationship Analysis

In [ ]:
df_modern['LogVolume'] = np.log1p(df_modern['ShareVolume'])
df_modern['AbsReturn'] = np.abs(df_modern['DailyReturn'])

plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_modern.sample(10000), x='LogVolume', y='AbsReturn', alpha=0.1, color=CSE_COLORS['primary'])
plt.title('Log Volume vs Absolute Daily Return (10k Sample)')
plt.xlabel('Log(Share Volume)')
plt.ylabel('Absolute Daily Return')
plt.tight_layout()
plt.show()

> **Insight - Relationships**: There is a visible "volatility smile" effect where higher trading volumes are associated with larger absolute price movements.

## Section 12: Time-Based Feature Engineering
Analyzing calendar effects.

In [ ]:
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['DayOfWeek'] = df['Date'].dt.dayofweek

dow_returns = df.groupby('DayOfWeek')['DailyReturn'].mean() * 100
days = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri']

plt.figure(figsize=(8, 5))
sns.barplot(x=days, y=dow_returns[:5], color=CSE_COLORS['teal'])
plt.title('Average Daily Return by Day of Week (%)')
plt.ylabel('Mean Return (%)')
plt.axhline(0, color='black', linewidth=1)
plt.tight_layout()
plt.show()

monthly_returns = df.groupby('Month')['DailyReturn'].mean() * 100
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

plt.figure(figsize=(10, 5))
sns.barplot(x=months, y=monthly_returns, color=CSE_COLORS['accent1'])
plt.title('Average Daily Return by Month (%)')
plt.ylabel('Mean Return (%)')
plt.axhline(0, color='black', linewidth=1)
plt.tight_layout()
plt.show()

> **Insight - Calendar Effects**: Minor seasonality is visible, but the effects are generally small and require rigorous statistical testing before being used as standalone signals.

## Section 13: Volatility Analysis

In [ ]:
# Annualized volatility per stock
ann_vol = df.groupby('CompanyCode')['DailyReturn'].std() * np.sqrt(252)
ann_vol = ann_vol.dropna().sort_values(ascending=False)

print("Top 10 Most Volatile Stocks (Annualized):")
display(ann_vol.head(10))

if not df_indices.empty:
    df_indices['Vol_30d'] = df_indices['Daily_Return'].rolling(30).std() * np.sqrt(252)
    
    plt.figure(figsize=(14, 5))
    plt.plot(df_indices.index, df_indices['Vol_30d'], color=CSE_COLORS['negative'])
    add_crash_bands(plt.gca())
    plt.title('ASPI 30-Day Rolling Annualized Volatility')
    plt.ylabel('Volatility')
    plt.tight_layout()
    plt.show()

> **Insight - Volatility**: Volatility exhibits strong clustering, spiking massively during the designated crash periods (e.g., 2022 Economic Crisis).

## Section 14: Trend Analysis

In [ ]:
if not df_indices.empty:
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.plot(df_indices.index, df_indices['ASPI'], color='black', label='ASPI', alpha=0.5)
    
    mas = [50, 200]
    colors = [CSE_COLORS['teal'], CSE_COLORS['orange']]
    
    for ma, color in zip(mas, colors):
        df_indices[f'MA_{ma}'] = df_indices['ASPI'].rolling(ma).mean()
        ax.plot(df_indices.index, df_indices[f'MA_{ma}'], color=color, label=f'{ma}-day MA', linewidth=2)
        
    ax.set_title('ASPI Trend Analysis (50 & 200 Day MAs)')
    ax.legend()
    plt.tight_layout()
    plt.show()

> **Insight - Trends**: The 50/200 day moving average crossovers provide clear historical signals for major market regime shifts.

## Section 15: Interactive Visualizations (Plotly)
*(Note: These will render in a live notebook environment)*

In [ ]:
if not df_indices.empty:
    # Downsample for plotly performance if needed
    plot_df = df_indices.reset_index().dropna(subset=['ASPI'])
    
    fig = px.line(plot_df, x='Date', y='ASPI', title='Interactive ASPI Chart')
    fig.update_xaxes(rangeslider_visible=True)
    # fig.show()  # Uncomment to view interactively
    print("Interactive ASPI chart generated (uncomment fig.show() to view in notebook).")

> **Insight - Interactivity**: Plotly allows for deep dives into specific crash and rally dates.

## Section 16: Closing Summary

### Key Findings
1. **Market Structure**: The dataset successfully bridges two eras (1991-2000 Close-only, and 2001-2025 OHLCV).
2. **Data Quality**: Identified missing values, zero prices, and potential unadjusted splits that require a robust cleaning pipeline before modeling.
3. **Distributions**: Stock returns show classic fat tails (excess kurtosis); standard normal assumptions will underestimate tail risk.
4. **Volatility Clustering**: Clear volatility regimes map exactly to Sri Lanka's macroeconomic and political events (2001, 2008, 2011, 2019, 2022).
5. **Liquidity**: Trading is heavily concentrated in a top tier of liquid stocks.

### Candidate Target Variables for Future Modeling
- `forward_1M_return`, `forward_3M_return` (for regression mapping to investment horizons)
- `trend_state` (binary label based on price position vs MAs)

### Next Steps (Phase 2)
- Implement a corporate action pipeline to adjust historical prices for stock splits using the `df_splits` table.
- Engineer technical indicators (RSI, MACD, Bollinger Bands) using the clean OHLCV data.
- Develop the filtering logic for the Recommendation System to map these features to Short/Medium/Long term uptrend potential.